<a href="https://colab.research.google.com/github/chethana-152005/Yuvaintern-Logistics-Data-Analyst-Intern/blob/main/task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

# Load the simulated logistics dataset
df = pd.read_csv('Food_Delivery_Time_Prediction.csv')

# Initial Data Inspection
print("Dataset Shape:", df.shape)
print("\nMissing Values per Column:\n", df.isnull().sum())

# Simulate some missing values for demonstration (if dataset is perfectly clean)
df.loc[10:15, 'Rider_Rating'] = np.nan
df.loc[20:25, 'Weather'] = np.nan

Dataset Shape: (50000, 24)

Missing Values per Column:
 Order_ID                      0
Order_Date                    0
Order_Hour                    0
Day_of_Week                   0
Is_Weekend                    0
Is_Festival                   0
Weather                       0
Pickup_Zone                   0
Dropoff_Zone                  0
Vehicle_Type                  0
Rider_Experience_Years        0
Rider_Rating                  0
Restaurant_Rating             0
Cuisine_Type                  0
Order_Items                   0
Restaurant_Load               0
Preparation_Time_Min          0
Road_Distance_km              0
Delivery_Distance_Category    0
Traffic_Level                 0
Number_of_Signals             0
Average_Speed_kmph            0
Delivery_Priority             0
Time_taken_min                0
dtype: int64


In [3]:
# 1. Handling Missing Values
# Impute numerical column with median
df['Rider_Rating'].fillna(df['Rider_Rating'].median(), inplace=True)
# Impute categorical column with mode
df['Weather'].fillna(df['Weather'].mode()[0], inplace=True)

# 2. Outlier Detection and Capping using IQR for 'Time_taken_min'
Q1 = df['Time_taken_min'].quantile(0.25)
Q3 = df['Time_taken_min'].quantile(0.75)
IQR = Q3 - Q1

# Define bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Cap outliers to the upper and lower bounds
df['Time_taken_min'] = np.where(df['Time_taken_min'] > upper_bound, upper_bound,
                       np.where(df['Time_taken_min'] < lower_bound, lower_bound, df['Time_taken_min']))

/tmp/ipykernel_911/526800692.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Rider_Rating'].fillna(df['Rider_Rating'].median(), inplace=True)
/tmp/ipykernel_911/526800692.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inpl

In [4]:
from sklearn.preprocessing import MinMaxScaler

# 3. Categorical Encoding (One-Hot Encoding)
categorical_cols = ['Weather', 'Traffic_Level', 'Pickup_Zone', 'Vehicle_Type']
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# 4. Feature Normalization (Min-Max Scaling)
numerical_cols_to_scale = ['Road_Distance_km', 'Average_Speed_kmph', 'Preparation_Time_Min']
scaler = MinMaxScaler()

df_encoded[numerical_cols_to_scale] = scaler.fit_transform(df_encoded[numerical_cols_to_scale])

print("\nData after Cleaning, Encoding, and Normalization:")
print(df_encoded.head())


Data after Cleaning, Encoding, and Normalization:
    Order_ID  Order_Date  Order_Hour Day_of_Week  Is_Weekend  Is_Festival  \
0  ORD000001  2025-10-28           8     Tuesday           0            0   
1  ORD000002  2025-09-26          12      Friday           0            0   
2  ORD000003  2025-01-26          18      Sunday           1            0   
3  ORD000004  2025-11-19          12   Wednesday           0            0   
4  ORD000005  2025-08-04          18      Monday           0            0   

  Dropoff_Zone  Rider_Experience_Years  Rider_Rating  Restaurant_Rating  ...  \
0   Commercial                     3.1           3.8                3.8  ...   
1          CBD                     1.8           3.6                3.9  ...   
2  Residential                     3.8           3.9                4.0  ...   
3   Industrial                     1.7           3.9                4.6  ...   
4  Residential                    12.4           4.4                4.0  ...   

  Tra